In [ ]:
# ============================================================
# ALEXNET + BAYESIAN OPTIMIZATION (FAST VERSION)
# ============================================================

import os, cv2, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import keras_tuner as kt

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/pld-foldds/PLD_FOLDDATASET"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD DATA ----------------
def load_data(folder):
    paths, labels = [], []
    classes = sorted(os.listdir(folder))

    for idx, cls in enumerate(classes):
        for f in os.listdir(os.path.join(folder, cls)):
            paths.append(os.path.join(folder, cls, f))
            labels.append(idx)

    return np.array(paths), np.array(labels), classes

X_train, y_train, class_names = load_data(TRAIN_DIR)
X_val, y_val, _ = load_data(VAL_DIR)
X_test, y_test, _ = load_data(TEST_DIR)

NUM_CLASSES = len(class_names)

# ---------------- GENERATOR ----------------
def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment and np.random.rand() < 0.5:
                    img = cv2.flip(img, 1)

                img = img.astype("float32") / 255.0
                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ---------------- MODEL BUILDER ----------------
def build_model(hp):

    model = models.Sequential()

    model.add(layers.Conv2D(96, (11,11), strides=4, activation='relu',
                            input_shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(3,2))

    model.add(layers.Conv2D(256, (5,5), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(3,2))

    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(3,2))

    model.add(layers.Flatten())

    # 🔥 Tunable dense layer
    dense_units = hp.Int("dense_units", 512, 1024, step=256)
    model.add(layers.Dense(dense_units, activation='relu'))

    # 🔥 Tunable dropout
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    model.add(layers.Dropout(dropout))

    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # 🔥 Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ---------------- TUNER ----------------
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=3,   # 🔥 as requested
    directory="/kaggle/working/tuner",
    project_name="alexnet_bayes"
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

# ---------------- SEARCH ----------------
tuner.search(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=len(X_train)//BATCH_SIZE,
    validation_data=generator(X_val, y_val),
    validation_steps=len(X_val)//BATCH_SIZE,
    epochs=5,
    callbacks=[early_stop]
)

# ---------------- BEST MODEL ----------------
best_model = tuner.get_best_models(1)[0]

print("✅ Best hyperparameters selected")

# ---------------- FINAL TRAIN ----------------
history = best_model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=len(X_train)//BATCH_SIZE,
    validation_data=generator(X_val, y_val),
    validation_steps=len(X_val)//BATCH_SIZE,
    epochs=15
)

# ---------------- SAVE ----------------
SAVE_PATH = "/kaggle/working/alexnet_bayes.h5"
best_model.save(SAVE_PATH)

ZIP_PATH = "/kaggle/working/alexnet_bayes.zip"
with zipfile.ZipFile(ZIP_PATH, 'w') as z:
    z.write(SAVE_PATH, arcname="alexnet_bayes.h5")

print("📦 Model saved & zipped")

# ---------------- TEST ----------------
y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(test_gen)
    preds = best_model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

print("\n✅ Test Accuracy:", np.mean(y_true == y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ============================================================
# DOMAIN ADAPTATION + FINE-TUNING (ALEXNET - FINAL FIXED)
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/potatoleafplantvillage/PlantVillageExtended/Test"

IMG_SIZE = 224   # ✅ IMPORTANT (AlexNet compatible)
BATCH_SIZE = 32
EPOCHS = 10

# ---------------- LOAD MODEL ----------------
MODEL_PATH = "/kaggle/working/alexnet_bayes.h5"
model = load_model(MODEL_PATH)

# ---------------- CLASS ORDER ----------------
class_names = ["EarlyBlight", "Healthy", "LateBlight"]

# ============================================================
# ROBUST DATA LOADING (AUTO MATCHING)
# ============================================================

paths, labels = [], []

for cls in os.listdir(DATASET_DIR):

    cls_lower = cls.lower()

    if "early" in cls_lower:
        label = class_names.index("EarlyBlight")
    elif "late" in cls_lower:
        label = class_names.index("LateBlight")
    elif "healthy" in cls_lower:
        label = class_names.index("Healthy")
    else:
        print("Skipping:", cls)
        continue

    cls_dir = os.path.join(DATASET_DIR, cls)

    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg','.png','.jpeg')):
            paths.append(os.path.join(cls_dir, f))
            labels.append(label)

paths = np.array(paths)
labels = np.array(labels)

print("✅ Total loaded samples:", len(paths))

if len(paths) == 0:
    raise ValueError("❌ No data loaded. Check dataset path.")

# ============================================================
# SPLIT (DOMAIN ADAPTATION SETUP)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    paths, labels,
    test_size=0.8,
    stratify=labels,
    random_state=42
)

print("Adaptation train:", len(X_train))
print("Test:", len(X_test))

# ============================================================
# DATA GENERATOR (FIXED FOR ALEXNET)
# ============================================================

def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                # 🔥 AUGMENTATION
                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)
                    if np.random.rand() < 0.3:
                        img = cv2.convertScaleAbs(img, alpha=1.2, beta=25)
                    if np.random.rand() < 0.3:
                        img = cv2.GaussianBlur(img, (3,3), 0)

                # ✅ CORRECT NORMALIZATION
                img = img.astype("float32") / 255.0

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, 3)

# ============================================================
# FINE-TUNING (SAFE FOR ALEXNET)
# ============================================================

# Only unfreeze last layers (avoid instability)
for layer in model.layers[-10:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# TRAINING
# ============================================================

history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# TESTING
# ============================================================

y_true, y_pred = [], []
gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ============================================================
# RESULTS
# ============================================================

acc = np.mean(y_true == y_pred)
print(f"\n✅ Improved Accuracy: {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ============================================================
# VGG19 PIPELINE (256x256) + FULL EVALUATION
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/pld-foldds/PLD_FOLDDATASET"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256   # ✅ as requested
BATCH_SIZE = 32
EPOCHS = 10

# ---------------- LOAD DATA ----------------
def load_data(folder):
    paths, labels = [], []
    classes = sorted(os.listdir(folder))

    for idx, cls in enumerate(classes):
        for f in os.listdir(os.path.join(folder, cls)):
            paths.append(os.path.join(folder, cls, f))
            labels.append(idx)

    return np.array(paths), np.array(labels), classes

X_train, y_train, class_names = load_data(TRAIN_DIR)
X_val, y_val, _ = load_data(VAL_DIR)
X_test, y_test, _ = load_data(TEST_DIR)

NUM_CLASSES = len(class_names)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)

                img = preprocess_input(img.astype("float32"))
                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ---------------- MODEL (VGG19) ----------------
base = VGG19(weights="imagenet", include_top=False,
             input_shape=(IMG_SIZE, IMG_SIZE, 3))

base.trainable = False  # freeze initially

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs=base.input, outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ---------------- TRAIN ----------------
history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=len(X_train)//BATCH_SIZE,
    validation_data=generator(X_val, y_val),
    validation_steps=len(X_val)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ---------------- TEST ----------------
y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(test_gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ---------------- METRICS ----------------
acc = np.mean(y_true == y_pred)
print(f"\n✅ Test Accuracy: {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ---------------- TRAINING CURVES ----------------
plt.figure(figsize=(12,4))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ================= SAVE MODEL =================

SAVE_PATH = "/kaggle/working/vgg19_model.h5"

model.save(SAVE_PATH)

print(f"✅ Model saved at: {SAVE_PATH}")

In [ ]:
# ============================================================
# DOMAIN ADAPTATION + FINE-TUNING (VGG19 VERSION)
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.vgg19 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/potatoleafplantvillage/PlantVillageExtended/Test"
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 10

# ---------------- LOAD MODEL ----------------
MODEL_PATH = "/kaggle/working/vgg19_model.h5"   # ✅ ensure this is VGG19 model
model = load_model(MODEL_PATH)

# ---------------- CLASS ORDER ----------------
class_names = ["EarlyBlight", "Healthy", "LateBlight"]

# ============================================================
# ROBUST DATA LOADING
# ============================================================

paths, labels = [], []

for cls in os.listdir(DATASET_DIR):

    cls_lower = cls.lower()

    if "early" in cls_lower:
        label = class_names.index("EarlyBlight")
    elif "late" in cls_lower:
        label = class_names.index("LateBlight")
    elif "healthy" in cls_lower:
        label = class_names.index("Healthy")
    else:
        print("Skipping:", cls)
        continue

    cls_dir = os.path.join(DATASET_DIR, cls)

    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg','.png','.jpeg')):
            paths.append(os.path.join(cls_dir, f))
            labels.append(label)

paths = np.array(paths)
labels = np.array(labels)

print("✅ Total loaded samples:", len(paths))

if len(paths) == 0:
    raise ValueError("❌ No data loaded")

# ============================================================
# SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    paths, labels,
    test_size=0.8,
    stratify=labels,
    random_state=42
)

print("Adaptation train:", len(X_train))
print("Test:", len(X_test))

# ============================================================
# GENERATOR (VGG19 PREPROCESSING)
# ============================================================

def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)
                    if np.random.rand() < 0.3:
                        img = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
                    if np.random.rand() < 0.3:
                        img = cv2.GaussianBlur(img, (3,3), 0)

                # ✅ VGG19 preprocessing
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, 3)

# ============================================================
# FINE-TUNING (SAFE)
# ============================================================

# Unfreeze only last few layers (recommended)
for layer in model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# TEST
# ============================================================

y_true, y_pred = [], []
gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ============================================================
# RESULTS
# ============================================================

acc = np.mean(y_true == y_pred)
print(f"\n✅ Improved Accuracy: {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ============================================================
# RESNET50 PIPELINE (256x256) + FULL EVALUATION + SAVE MODEL
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/pld-foldds/PLD_FOLDDATASET"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 10

# ---------------- LOAD DATA ----------------
def load_data(folder):
    paths, labels = [], []
    classes = sorted(os.listdir(folder))

    for idx, cls in enumerate(classes):
        for f in os.listdir(os.path.join(folder, cls)):
            paths.append(os.path.join(folder, cls, f))
            labels.append(idx)

    return np.array(paths), np.array(labels), classes

X_train, y_train, class_names = load_data(TRAIN_DIR)
X_val, y_val, _ = load_data(VAL_DIR)
X_test, y_test, _ = load_data(TEST_DIR)

NUM_CLASSES = len(class_names)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)

                img = preprocess_input(img.astype("float32"))
                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ---------------- MODEL (ResNet50) ----------------
base = ResNet50(weights="imagenet", include_top=False,
                input_shape=(IMG_SIZE, IMG_SIZE, 3))

base.trainable = False  # initial freeze

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs=base.input, outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ---------------- CALLBACKS (SAVE BEST MODEL) ----------------
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/best_resnet50.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# ---------------- TRAIN ----------------
history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    validation_data=generator(X_val, y_val),
    validation_steps=max(1, len(X_val)//BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=[checkpoint],
    verbose=1
)

# ---------------- SAVE FINAL MODEL ----------------
model.save("/kaggle/working/final_resnet50.h5")
print("✅ Final model saved")

# ---------------- LOAD BEST MODEL ----------------
model = tf.keras.models.load_model("/kaggle/working/best_resnet50.h5")

# ---------------- TEST ----------------
y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(test_gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ---------------- METRICS ----------------
acc = np.mean(y_true == y_pred)
print(f"\n✅ Test Accuracy: {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ---------------- TRAINING CURVES ----------------
plt.figure(figsize=(12,4))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# OPTIMIZED DOMAIN ADAPTATION (RESNET50 - HIGH ACCURACY)
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/potatoleafplantvillage/PlantVillageExtended/Test"  # ✅ FIXED
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 15

class_names = ["EarlyBlight", "Healthy", "LateBlight"]

# ============================================================
# DATA LOADING
# ============================================================

paths, labels = [], []

for cls in os.listdir(DATASET_DIR):
    cls_lower = cls.lower()

    if "early" in cls_lower:
        label = 0
    elif "healthy" in cls_lower:
        label = 1
    elif "late" in cls_lower:
        label = 2
    else:
        continue

    cls_dir = os.path.join(DATASET_DIR, cls)

    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg','.png','.jpeg')):
            paths.append(os.path.join(cls_dir, f))
            labels.append(label)

paths = np.array(paths)
labels = np.array(labels)

print("Total samples:", len(paths))

# ============================================================
# SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.3, stratify=labels, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# ============================================================
# STRONG AUGMENTATION
# ============================================================

def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)
                    if np.random.rand() < 0.4:
                        img = cv2.convertScaleAbs(img, alpha=1.4, beta=40)
                    if np.random.rand() < 0.3:
                        img = cv2.GaussianBlur(img, (5,5), 0)
                    if np.random.rand() < 0.3:
                        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
                    if np.random.rand() < 0.2:
                        img = cv2.add(img, np.random.randint(0,30,img.shape,dtype='uint8'))

                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, 3)

# ============================================================
# MODEL
# ============================================================

base = ResNet50(weights="imagenet",
                include_top=False,
                input_shape=(IMG_SIZE, IMG_SIZE, 3))

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(3, activation="softmax")(x)

model = Model(inputs=base.input, outputs=outputs)

# ============================================================
# BETTER FINE-TUNING
# ============================================================

for layer in base.layers[:-50]:
    layer.trainable = False

for layer in base.layers[-50:]:
    layer.trainable = True

# ============================================================
# COSINE LR SCHEDULE
# ============================================================

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=1000
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

# ============================================================
# CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_resnet50.h5",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    validation_data=generator(X_val, y_val),
    validation_steps=max(1, len(X_val)//BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# LOAD BEST MODEL
# ============================================================

model = tf.keras.models.load_model("/kaggle/working/best_resnet50.h5")

# ============================================================
# TEST
# ============================================================

y_true, y_pred = [], []
gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ============================================================
# RESULTS
# ============================================================

print("\n✅ Accuracy:", np.mean(y_true == y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ============================================================
# DENSENET201 PIPELINE (256x256) + FULL EVALUATION + OPTIMIZED
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/datasets/psthakre/pld-foldds/PLD_FOLDDATASET"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 10   # 🔥 increased

# ---------------- LOAD DATA ----------------
def load_data(folder):
    paths, labels = [], []
    classes = sorted(os.listdir(folder))

    for idx, cls in enumerate(classes):
        for f in os.listdir(os.path.join(folder, cls)):
            paths.append(os.path.join(folder, cls, f))
            labels.append(idx)

    return np.array(paths), np.array(labels), classes

X_train, y_train, class_names = load_data(TRAIN_DIR)
X_val, y_val, _ = load_data(VAL_DIR)
X_test, y_test, _ = load_data(TEST_DIR)

NUM_CLASSES = len(class_names)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

# ============================================================
# STRONG GENERATOR
# ============================================================

def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)
                    if np.random.rand() < 0.4:
                        img = cv2.convertScaleAbs(img, alpha=1.4, beta=40)
                    if np.random.rand() < 0.3:
                        img = cv2.GaussianBlur(img, (5,5), 0)
                    if np.random.rand() < 0.3:
                        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# MODEL (DenseNet201)
# ============================================================

base = DenseNet201(weights="imagenet",
                   include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))

base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)   # 🔥 added
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs=base.input, outputs=out)

# ============================================================
# FINE-TUNING
# ============================================================

for layer in base.layers[:-60]:
    layer.trainable = False

for layer in base.layers[-60:]:
    layer.trainable = True

# ============================================================
# LR + LOSS OPTIMIZATION
# ============================================================

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=1000
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# CALLBACKS
# ============================================================

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/best_densenet201.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

callbacks = [
    checkpoint,
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    validation_data=generator(X_val, y_val),
    validation_steps=max(1, len(X_val)//BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# SAVE FINAL MODEL
# ============================================================

model.save("/kaggle/working/final_densenet201.h5")

# ============================================================
# LOAD BEST MODEL
# ============================================================

model = tf.keras.models.load_model("/kaggle/working/best_densenet201.h5")

# ============================================================
# TEST
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(test_gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ============================================================
# RESULTS
# ============================================================

acc = np.mean(y_true == y_pred)
print(f"\n✅ Test Accuracy: {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ============================================================
# TRAINING CURVES
# ============================================================

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# CLEAN DOMAIN ADAPTATION (LEAKAGE-FREE)
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/PlantVillageExtended"
MODEL_PATH = "/kaggle/working/best_model_A.h5"   # trained on Dataset A

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 10

class_names = ["EarlyBlight", "Healthy", "LateBlight"]

# ============================================================
# LOAD DATASET B
# ============================================================

paths, labels = [], []

for cls in os.listdir(DATASET_DIR):
    cls_lower = cls.lower()

    if "early" in cls_lower:
        label = 0
    elif "healthy" in cls_lower:
        label = 1
    elif "late" in cls_lower:
        label = 2
    else:
        continue

    cls_dir = os.path.join(DATASET_DIR, cls)

    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg','.png','.jpeg')):
            paths.append(os.path.join(cls_dir, f))
            labels.append(label)

paths = np.array(paths)
labels = np.array(labels)

print("Total samples:", len(paths))

# ============================================================
# SPLIT (IMPORTANT)
# ============================================================

# 70% TEST (UNSEEN)
X_temp, X_test, y_temp, y_test = train_test_split(
    paths, labels,
    test_size=0.7,
    stratify=labels,
    random_state=42
)

# Remaining 30% → split into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.33,   # ~10% val
    stratify=y_temp,
    random_state=42
)

print("Adapt Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

# ============================================================
# GENERATOR
# ============================================================

def generator(paths, labels, augment=False):
    while True:
        idxs = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                if augment:
                    if np.random.rand() < 0.5:
                        img = cv2.flip(img, 1)
                    if np.random.rand() < 0.4:
                        img = cv2.convertScaleAbs(img, alpha=1.3, beta=40)
                    if np.random.rand() < 0.3:
                        img = cv2.GaussianBlur(img, (5,5), 0)

                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, 3)

# ============================================================
# LOAD MODEL TRAINED ON DATASET A
# ============================================================

model = load_model(MODEL_PATH)

# ============================================================
# FINE-TUNING
# ============================================================

# Freeze most layers
for layer in model.layers[:-40]:
    layer.trainable = False

# Unfreeze last layers
for layer in model.layers[-40:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

# ============================================================
# CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_adapted_model.h5",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

# ============================================================
# TRAIN (ADAPTATION)
# ============================================================

history = model.fit(
    generator(X_train, y_train, augment=True),
    steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
    validation_data=generator(X_val, y_val),
    validation_steps=max(1, len(X_val)//BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# LOAD BEST MODEL
# ============================================================

model = load_model("/kaggle/working/best_adapted_model.h5")

# ============================================================
# TEST (UNSEEN DATA)
# ============================================================

y_true, y_pred = [], []
gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE + 1):
    imgs, labs = next(gen)
    preds = model.predict(imgs, verbose=0)

    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true[:len(X_test)])
y_pred = np.array(y_pred[:len(X_test)])

# ============================================================
# RESULTS
# ============================================================

acc = np.mean(y_true == y_pred)
print(f"\n✅ Final Test Accuracy (Domain Adaptation): {acc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()